In [1]:
import os
import glob
from sunscc.nb.load import load_from_dir, load_from_dir2, load_from_dir_new_pl, load_from_cfg
from pathlib import Path
from astropy.io import fits
import skimage.io as io
import numpy as np
import torch 
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
%matplotlib ipympl

from sunscc.dataset import DeepsunSegmentationPredictDataset, get_grid_size, crop_patch
from sunscc.dataset.utils import *
from sunscc.callback import reconstruct_segmentation_image, display_classification_predictions

import ipywidgets as widgets

`torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.


### Set environment variable and path to the image to be processed

In [2]:
cwd = Path(os.getcwd())
tmp = cwd.parents[0]
os.environ['SUNSCCDATASET_PATH'] = str(tmp)
print(os.environ['SUNSCCDATASET_PATH'])

input_dir = Path('./inputs/')
input_files = sorted(glob.glob(str(input_dir / '*')))
print(input_files)

output_dir = Path('./predictions_segmentation')
if not output_dir.exists():
    output_dir.mkdir()

/home/rwc/sunscc_repo/sunscc-rob
['inputs/UPH20240510080302 copy 2.FTS', 'inputs/UPH20240510080302 copy.FTS', 'inputs/UPH20240510080302.FTS']


# Segmentation

### Define path to Segmentation model and load it.

In [3]:
segmenter_run_dir = Path(os.environ['SUNSCCDATASET_PATH']) / 'outputs/SunSCC_Model' / 'Segmentation/seg_run0'

In [4]:
dataset_overrides = [
        'dataset.test_dataset.transforms=[]', 
    ]
seg_config, seg_model, seg_dm, seg_trainer = load_from_dir_new_pl(
                                    run_path= segmenter_run_dir.resolve(),
                                    # run_path= run_dir,
                                    load_trainer=False,
                                    override = dataset_overrides + [
                                        # Don't create new wandb run when testing/predicting
#                                         f'logger=[{{_target_: pytorch_lightning.loggers.CSVLogger, save_dir: "csv_metrics"}}]',
                                        '~callbacks.LR_monitor',
                                        '~callbacks.wandb',
                                    ]
                                )
# dm.setup()
test_dataloader = seg_dm.test_dataloader()


The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
Lightning automatically upgraded your loaded checkpoint from v1.6.0 to v2.4.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../data/usetml/SunSCC_Model/Segmentation/seg_run0/models/last.ckpt`


BaseSegment
{'_target_': 'sunscc.model.ConfUnet', 'model_cfg': {'_target_': 'sunscc.model.ModelConfig'}, 'architecture': {'first': 32, 'enc': {'width': [32, 64, 128, 256, 512, 512, 512], 'repeat': [1, 1, 1, 1, 1, 1, 1]}, 'dec': {'width': [512, 512, 256, 128, 64, 32], 'repeat': [1, 1, 1, 1, 1, 1]}}, 'input_format': ['image'], 'output_format': ['segmentation'], 'classes': ['sunspot'], 'block': 'double'}
sunscc.loss.CombineLosses
Ignored_indexA: -100  Ignored_indexB: None
['sunspot']


### Load images in input directory. For each, split it in a grid and create a batch for segmentation model

In [5]:
with torch.no_grad():
    for input_image in tqdm(input_files):

        hdulst:fits.HDUList = fits.open(input_image)
        image = hdulst[0]
        header = image.header
        center = np.array(image.shape)//2
        radius = header['SOLAR_R']

        patch_side = test_dataloader.dataset.patch_side
        grid_size = get_grid_size(input_image, patch_side)

        batch_v1 = {}

        for index in range(grid_size):

            sample = {} 
            idx_img = index // grid_size
            idx_patch = index % grid_size

            sample["image"] = crop_patch((image.data).astype(float), idx_patch, patch_side, grid_size) # load image from directory with skimage
            sample['sample_id'] = f'{idx_img}_{idx_patch}'
            sample["name"] = os.path.basename(input_image)

            sample["segmentation"] = np.zeros_like(sample["image"])
            
            batch_v1[idx_patch] = sample

        hdulst.close()

        keys_to_tensor = ['image', 'segmentation']

        batch_v2 = {key: torch.tensor(np.concatenate([d[key][None,:,:] for d in batch_v1.values()])) 
                    if key in keys_to_tensor else [d[key] for d in batch_v1.values()] 
                    for key in batch_v1[0].keys()}
        
        ### Predict the segmentation masks, reconstruct complete image, write on disk
        output_seg = seg_model.predict(batch_v2)

        reconstruct_segmentation_image(grid_size, batch_v2, output_seg, output_dir)
        


  0%|          | 0/3 [00:00<?, ?it/s]

###  Load segmentation mask and show

In [7]:
def refresh(_):
    idx = slider.value
    input_image = input_files[idx]
    seg_mask_fn = output_dir/(os.path.basename(input_image).split(".")[0]+ ".png")
    seg_mask = io.imread(seg_mask_fn)

    ax.imshow(image.data, cmap='gray', interpolation=None)
    ax.set_title(f'Image w/o segmentation mask')
    if mask_check.value:
        ax.imshow(seg_mask, cmap='jet', alpha=0.5 )
        ax.set_title(f'Image with segmentation mask')
    plt.draw()


slider = widgets.IntSlider(value=0, min=0, max=len(input_files)-1, step=1, description='Patch index:', continuous_update=False)
slider.observe(refresh, names='value')

mask_check = widgets.Checkbox(value=True, description='Show mask')
mask_check.observe(refresh, names='value')

plt.ioff()
fig, ax = plt.subplots(1, 1, figsize=(4,4))
refresh(None)
plt.ion()

display(widgets.VBox([widgets.HBox([slider,mask_check]) , fig.canvas]))